In [1]:
import json

In [2]:
import numpy

In [3]:
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [5]:
import pandas as pd

In [6]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [7]:
MODEL_ID = "google/gemma-4-E2B-it"

In [8]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

In [10]:
df = pd.read_excel("data.xlsx", header=0)

In [11]:
df.head()

,title_rus,title_eng,goals,tasks,annotation,description,expectations,product_result,result_criterias,social_effect,commercial_effect
0,Исследование приоритетов и механизмов реализац...,Study of Priorities and Mechanisms for Impleme...,Целью проекта является эмпирическая проверка г...,1) Анализ повестки международных доноров\n2) А...,Работа международных фондов (доноров) должна п...,"Согласно определению международных фондов, про...",Аналитический отчет по избранным странам.\n\nС...,NaN,NaN,NaN,NaN
1,Антрополе - научно-популярный видео-подкаст о ...,Anthropole is a Popular Science Video Podcast ...,Выпустить и популяризовать 27 видео-подкастов ...,Снять и смонтировать подкасты;\nРазработать ст...,"\tАнтрополе - научно-популярный проект, в рамк...","Социальное знание близко и интересно обществу,...","Студенты получат опыт монтажа, продвижения и к...",Регулярный видео-подкаст (+экспедиционные филь...,Опубликованы 27 видео-подкастов о необычных со...,Популяризация социального знания должна привес...,Планируется в течение года выйти на окупаемост...
2,"Разработка, создание и ведение сайта, посвящен...","Design, Development and Implementation of a We...","Для того, чтобы получить более полное представ...",Подготовка технического задания для разработчи...,Художественное образование и творчество художн...,Тема обучения арабских художников в художестве...,"Навыки создания сайта (полный цикл, от подгото...","Создание сайта, посвященного истории арабских ...",Выполнение заданий руководителя проекта,Укрепление международных связей с художниками ...,NaN
3,Перевод с английского языка коллективной моног...,Translation from English of the collective mon...,Результатом проекта станет качественный перево...,Результатом проекта станет качественный перево...,"Коллективная монография, авторы которой являют...","Коллективная монография, авторы которой являют...",Участники проекта приобретут новые знания в об...,Результатом проекта станет качественный перево...,Критериями достижения результата будет возможн...,NaN,NaN
4,Сеть военно-политических союзов в Евразии: баз...,Network of Military in Eurasia: a Database,Целью проекта является создание базы данных со...,1.\tОпределение методики включения союзов в ба...,Проект посвящен изучению сети военно-политичес...,Проект посвящен анализу истории существования ...,1.\tОбучение навыкам сбора и анализа данных 2....,База данных союзов 1815-2024 в Евразии.,Создание базы данных как минимум тысячи диадны...,NaN,NaN


In [12]:
df = df.fillna("")

In [13]:
df.drop_duplicates(inplace=True)

In [14]:
SYSTEM_PROMPT = """
You are a helpful assistant, which can create artificial user profiles based on student project titles in russian.
Please, analyze given list of student project titles in russian and create several possible user profiles with own preferences in various fields of science.
Return stricly JSON output.
For example.
```json
{
'monetary_policy_researcher': 'A user, which is interesed in macroeconimcs in general and monetary police studies in particular',
'computer_vision_researcher': 'A user, which is interested in machine learning and computer vision in particular'
}
```
"""

In [15]:
messages = [
    (
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps({"student_projects_titles" : list(numpy.random.choice([row[1]["title_rus"] for row in df.iterrows()], size=16, replace=False))})},
        )
]

In [16]:
len(messages[0][1]["content"])

5901

In [17]:
for message in messages:
    text = processor.apply_chat_template(
        message, 
        tokenize=False, 
        add_generation_prompt=True, 
        enable_thinking=False
    )
    
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    
    outputs = model.generate(**inputs, max_new_tokens=1024)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    
    import re
    json_match = re.search(r'```json\s*(.*?)\s*```', response, re.DOTALL)
    if json_match:
        json_str = json_match.group(1)
    else:
        json_str = response
    
    answer = json.loads(json_str)

In [18]:
len(answer)

6

In [19]:
answer

{'digital_humanities_researcher': 'A user interested in the intersection of digital humanities, historical text analysis, and potentially computational linguistics or data science applied to humanities subjects. They value deep textual analysis and digital preservation methods.',
 'speech_ai_developer': 'A user focused on Artificial Intelligence, specifically in Speech Recognition and Natural Language Processing (NLP), with an interest in developing and implementing Speech AI systems.',
 'academic_researcher_hse_gcii': 'A user focused on academic research, potentially in education technology or specific academic fields (like HSE GCII), with a strong interest in structured research methodologies and potentially advanced topics like cognitive science or educational data.',
 'crm_travel_tech_analyst': 'A user interested in Business Intelligence (CRM), Travel Technology, and B2B strategies, likely focusing on market analysis, customer relationship management, and tech integration in the tr

In [20]:
with open("artificial_profiles.json", "w") as f:
    json.dump(answer, f)